In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import multibind as mb
import numpy as np
import pandas as pd
import torch
import bindome as bd
bd.constants.ANNOTATIONS_DIRECTORY = 'annotations'
# mb.models.MultiBind
import torch.optim as topti
import torch.utils.data as tdata
import matplotlib.pyplot as plt
import logomaker
import copy

# Use a GPU if available, as it should be faster.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device: " + str(device))

In [ ]:
n_rounds = 1
data = pd.read_csv('../data/countTable.0.CTCF_r3.tsv.gz', sep='\t', header=None)
data.columns = ['seq'] + [i for i in range(n_rounds+1)]
# data = data.sample(n=1000)
# data.index = range(len(data))
dataset = mb.datasets.SelexDataset(data, n_rounds=n_rounds)
train = tdata.DataLoader(dataset=dataset, batch_size=256, shuffle=True)
train_test = tdata.DataLoader(dataset=dataset, batch_size=1, shuffle=False)

In [ ]:
model = mb.models.DinucSelex(use_dinuc=False, kernels=[0, 14, 12], n_rounds=1, n_batches=1).to(device)
l2 = []
optimiser = topti.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
# optimiser = topti.LBFGS(model.parameters())
criterion = mb.tl.PoissonLoss()

In [ ]:
model1 = copy.deepcopy(model)
optimiser1 = topti.Adam(model1.parameters(), lr=0.001, weight_decay=0.0001)

In [ ]:
mb.tl.train_network(model1, train, device, optimiser1, criterion, dirichlet_regularization=0, num_epochs=500, early_stopping=10, log_each=10)

In [ ]:
mb.pl.conv_mono(model1)

In [ ]:
model2 = copy.deepcopy(model)
optimiser2 = topti.Adam(model2.parameters(), lr=0.001, weight_decay=0.0001)

In [ ]:
mb.tl.train_network(model2, train, device, optimiser2, criterion, dirichlet_regularization=0.002, num_epochs=500, early_stopping=10, log_each=10)

In [ ]:
mb.pl.conv_mono(model2)

In [ ]:
model3 = copy.deepcopy(model)
optimiser3 = topti.Adam(model3.parameters(), lr=0.001, weight_decay=0.0001)

In [ ]:
mb.tl.train_network(model3, train, device, optimiser3, criterion, dirichlet_regularization=0.0002, num_epochs=500, early_stopping=10, log_each=10)

In [ ]:
mb.pl.conv_mono(model3)